# Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis on the cleaned Seattle Airbnb dataset.

**Objectives:**
- Understand price patterns across neighborhoods
- Analyze key predictors (accommodates, bedrooms, property type)
- Identify relationships for hierarchical modeling
- Validate assumptions for Bayesian framework

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

%matplotlib inline

## 1. Load Cleaned Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/processed/listings_clean.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

df.head()

## 2. Price Distribution Analysis

In [ ]:
# Analyze price distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Raw prices
axes[0, 0].hist(df['price_clean'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].axvline(df['price_clean'].median(), color='red', linestyle='--', 
                   label=f'Median: ${df["price_clean"].median():.2f}')
axes[0, 0].legend()

# Log prices
log_prices = np.log(df['price_clean'])
axes[0, 1].hist(log_prices, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Log(Price)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Log Price Distribution')

# Q-Q plot for normality
stats.probplot(log_prices, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot: Log Prices vs Normal Distribution')

# Box plot
axes[1, 1].boxplot(df['price_clean'], vert=True)
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].set_title('Price Box Plot')

plt.tight_layout()
plt.show()

print(f"\n=== Log-Normality Test ===")
statistic, p_value = stats.shapiro(log_prices.sample(min(5000, len(log_prices))))
print(f"Shapiro-Wilk test p-value: {p_value:.4f}")
print(f"Log-normal assumption: {'✓ Valid' if p_value > 0.05 else '✗ Questionable'}")

## 3. Neighborhood-Level Analysis

In [ ]:
# Identify neighborhood column
neighborhood_col = 'neighbourhood_cleansed' if 'neighbourhood_cleansed' in df.columns else 'neighbourhood'

# Calculate neighborhood statistics
neighborhood_stats = df.groupby(neighborhood_col).agg({
    'price_clean': ['count', 'mean', 'std', 'median']
}).round(2)
neighborhood_stats.columns = ['Count', 'Mean', 'Std', 'Median']
neighborhood_stats = neighborhood_stats[neighborhood_stats['Count'] >= 10]  # Filter small neighborhoods
neighborhood_stats = neighborhood_stats.sort_values('Mean', ascending=False)

print(f"\n=== Top 10 Neighborhoods by Average Price ===")
print(neighborhood_stats.head(10))

# Visualize price variation across neighborhoods
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top neighborhoods bar chart
top_20 = neighborhood_stats.head(20)
top_20['Mean'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Average Price ($)')
axes[0].set_title('Top 20 Neighborhoods by Average Price')

# Coefficient of variation
neighborhood_stats['CV'] = neighborhood_stats['Std'] / neighborhood_stats['Mean']
top_cv = neighborhood_stats.nlargest(20, 'CV')
top_cv['CV'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Coefficient of Variation')
axes[1].set_title('Neighborhoods with Highest Price Variability')

plt.tight_layout()
plt.show()

## 4. Relationship: Accommodates vs Price

In [ ]:
# Analyze accommodates effect on price
if 'accommodates' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot
    axes[0].scatter(df['accommodates'], df['price_clean'], alpha=0.3, s=10)
    axes[0].set_xlabel('Number of Guests (Accommodates)')
    axes[0].set_ylabel('Price ($)')
    axes[0].set_title('Price vs Accommodates')
    
    # Box plot by accommodates
    df[df['accommodates'] <= 10].boxplot(column='price_clean', by='accommodates', ax=axes[1])
    axes[1].set_xlabel('Number of Guests')
    axes[1].set_ylabel('Price ($)')
    axes[1].set_title('Price Distribution by Accommodates')
    plt.suptitle('')  # Remove default title
    
    plt.tight_layout()
    plt.show()
    
    # Calculate correlation
    corr = df[['accommodates', 'price_clean']].corr().iloc[0, 1]
    print(f"\nCorrelation (accommodates vs price): {corr:.3f}")

## 5. Varying Slopes Analysis

Investigate if the effect of accommodates on price varies by neighborhood (justification for varying slopes model).

In [ ]:
# Analyze varying slopes by neighborhood
if 'accommodates' in df.columns:
    # Select top neighborhoods by count
    top_neighborhoods = df[neighborhood_col].value_counts().head(10).index
    df_top = df[df[neighborhood_col].isin(top_neighborhoods)]
    
    # Create log price
    df_top['log_price'] = np.log(df_top['price_clean'])
    
    # Plot regression lines for each neighborhood
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for neighborhood in top_neighborhoods:
        df_hood = df_top[df_top[neighborhood_col] == neighborhood]
        
        # Fit simple linear regression
        x = df_hood['accommodates'].values
        y = df_hood['log_price'].values
        
        # Remove NaN values
        mask = ~np.isnan(x) & ~np.isnan(y)
        x, y = x[mask], y[mask]
        
        if len(x) > 10:
            # Fit line
            z = np.polyfit(x, y, 1)
            p = np.poly1d(z)
            
            # Plot
            x_line = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_line, p(x_line), label=f'{neighborhood} (slope={z[0]:.3f})', linewidth=2)
    
    ax.set_xlabel('Number of Guests (Accommodates)')
    ax.set_ylabel('Log(Price)')
    ax.set_title('Varying Slopes by Neighborhood\n(Accommodates Effect on Log Price)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Interpretation ===")
    print("Different slopes indicate that the effect of accommodates on price")
    print("varies by neighborhood, justifying a varying slopes hierarchical model.")

## 6. Property Type Analysis

In [ ]:
# Analyze property types
if 'property_type' in df.columns:
    property_counts = df['property_type'].value_counts().head(10)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Property type distribution
    property_counts.plot(kind='barh', ax=axes[0], color='teal')
    axes[0].set_xlabel('Count')
    axes[0].set_title('Top 10 Property Types')
    
    # Average price by property type
    property_prices = df.groupby('property_type')['price_clean'].mean().sort_values(ascending=False).head(10)
    property_prices.plot(kind='barh', ax=axes[1], color='purple')
    axes[1].set_xlabel('Average Price ($)')
    axes[1].set_title('Average Price by Property Type')
    
    plt.tight_layout()
    plt.show()

## 7. Correlation Matrix

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col in ['price_clean', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'number_of_reviews']]

if len(numeric_cols) > 1:
    # Compute correlation matrix
    corr_matrix = df[numeric_cols].corr()
    
    # Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## Summary of EDA Findings

**Key Insights:**
1. **Log-normal prices**: Price distribution is approximately log-normal, justifying our modeling choice
2. **Neighborhood effects**: Significant variation in baseline prices across neighborhoods (varying intercepts)
3. **Varying slopes**: The effect of accommodates on price differs by neighborhood
4. **Strong predictors**: Accommodates, bedrooms, and property type are key price drivers

**Implications for Bayesian Model:**
- Use log-normal likelihood for price modeling
- Implement varying intercepts to capture neighborhood baseline differences
- Include varying slopes for accommodates effect
- Consider hierarchical structure to pool information across neighborhoods